# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

## Technical Interview Prep Tutor

Preparing for technical interviews across multiple analytics and AI roles can be overwhelming — 
each role requires different skills, knowledge and depth. This tool solves that by acting as a 
role-specific technical coach that tailors its answers based on the role you are preparing for.

### What this tool does
- Asks you to choose your target role before starting
- Acts as a specialized technical coach for that specific role
- Answers any technical question
- Connects answers back to your role when relevant
- Provides code examples where helpful
- Maintains full conversation history for follow up questions
- Supports both OpenAI GPT-4.1-mini and Qwen2.5-coder via Ollama

### Roles supported
1. Data Scientist
2. Machine Learning Engineer
3. Business Analyst
4. Data Analyst
5. AI/LLM Engineer

### Technical Implementation
- **Dual LLM Integration** — connects to OpenAI API (GPT-4.1-mini) and 
Ollama (Qwen2.5-coder) using the same OpenAI Python library by switching 
`base_url` — possible because Ollama uses an OpenAI compatible endpoint

- **Dynamic Prompt Engineering** — role specific system prompts stored in 
a Python dictionary and selected at runtime based on user input — same 
question gets completely different depth and focus per role

- **DRY Prompt Design** — shared `general_instruction` appended to every 
role prompt avoiding repetition — change once, updates all roles automatically

- **Multi-turn Conversation Management** — full conversation history 
maintained in a messages list and passed with every stateless API call — 
simulating stateful behavior exactly like ChatGPT does behind the scenes

- **Model Agnostic Architecture** — `chat()` function accepts client and 
model as parameters making it completely model agnostic — same function 
works for both cloud and local models without any changes

- **Input Validation** — quit checks at every user input point allowing 
graceful exit at any stage of the conversation flow

### How to use
1. Run all cells
2. Choose your model — GPT-4.1-mini (fast) or Qwen2.5-coder (free, local)
3. Choose your target role
4. Start asking technical questions!
5. Type 'quit' to end the session

### Requirements
- OpenAI API key in `.env` file
- Ollama installed locally with qwen2.5-coder pulled (for Ollama option)
- Install dependencies: `uv pip install openai python-dotenv`

---
*Built as a Week 1 Exercise for the Udemy course: AI Engineer Core Track — LLM Engineering, RAG, QLoRA, Agents by Ed Donner.*

In [ ]:
#Verify Ollama is running locally
import requests
requests.get("http://localhost:11434").content

In [ ]:
# Pull Qwen2.5-coder model from Ollama
!ollama pull qwen2.5-coder

In [ ]:
# Install required libraries
!uv pip install openai python-dotenv

In [ ]:
# Load environment variables
from dotenv import load_dotenv
import os

In [ ]:
# imports
from openai import OpenAI              # connects to Ollama using OpenAI compatible endpoint
from IPython.display import Markdown, display  # renders tutor responses as formatted Markdown

In [ ]:
# constants

MODEL_GPT = 'gpt-4.1-mini' 
MODEL_OLLAMA = 'qwen2.5-coder'

In [ ]:
# Load API key
load_dotenv(override=True)

# OpenAI client
openai_client = OpenAI()

# Ollama client
ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

### Prompt Design

This tool uses only **system prompts** — no separate user prompt is needed 
because the user's questions are passed directly as input in the conversation loop.

### General Instruction
A shared `general_instruction` string is appended to every role prompt. This ensures:
- Answers any technical question
- Answers are connected back to the role when relevant
- Code examples are always provided where helpful


### Role Specific System Prompts
A dictionary of 5 system prompts — one per role — that dynamically sets 
the tutor's expertise and focus area at the start of each session:
- Each prompt defines the tutor's role, focus areas and interview preparation strategy
- Selected at runtime based on user's role choice
- Combined with `general_instruction` for complete behavior definition

In [ ]:
general_instruction = """

You should answer ANY technical question the person asks — 
never refuse or redirect. If they ask about topics outside their 
role, answer fully and helpfully. When relevant, connect it back 
to how it applies to their specific role with a practical example.

Always be encouraging, clear and provide code examples where helpful.
"""

role_prompts = {
    "1": """You are a technical coach and an expert at answering 
questions for a Data Scientist role. Guide this person technically 
to ace their interview by explaining concepts clearly, providing 
relevant code examples, and focusing on skills that matter most 
for Data Science roles such as statistics, ML algorithms, 
Python, and model evaluation.""" + general_instruction,

    "2": """You are a technical coach and an expert at answering 
questions for a Machine Learning Engineer role. Guide this person 
technically to ace their interview by focusing on ML systems design, 
model deployment, MLOps, Python, and production ML concepts.""" + general_instruction,

    "3": """You are a technical coach and an expert at answering 
questions for a Business Analyst role. Guide this person technically 
to ace their interview by focusing on SQL, data visualization, 
requirements gathering, stakeholder communication and analytical 
thinking.""" + general_instruction,

    "4": """You are a technical coach and an expert at answering 
questions for a Data Analyst role. Guide this person technically 
to ace their interview by focusing on SQL, Excel, Python basics, 
data visualization, and statistical analysis.""" + general_instruction,

    "5": """You are a technical coach and an expert at answering 
questions for an AI/LLM Engineer role. Guide this person technically 
to ace their interview by focusing on LLMs, prompt engineering, 
RAG, fine tuning, agents, and Python.""" + general_instruction,
}

### Chat Function
The `chat()` function handles the entire conversation loop:
- Initializes the messages list with the role specific system prompt
- Runs a continuous `while True` loop accepting user questions
- Appends each question and answer to the messages list maintaining full conversation history
- Makes an API call with the complete messages list on every turn — simulating memory on top of stateless API calls
- Displays responses as formatted Markdown
- Exits gracefully when user types 'quit'

In [ ]:
def chat(model, client, role_choice):
    # 1. Set up messages list with system prompt
    messages = [
        {"role": "system", "content": role_prompts[role_choice]}
    ]
    

    # 2. Start conversation loop
    while True:
        # 4. Get user input
        user_input = input("You: ")
        
        # 5. Check if quit
        if user_input == "quit":
            print("Good luck with your interview!")
            break
            
        # 3. Append to messages
        messages.append({"role": "user", "content": user_input})
        print(f"\nYou: {user_input}")
        print("\nTutor:")
        
        # 4. Call API
        response = client.chat.completions.create(
            model=model,
            messages=messages, # entire conversation every time!
        )
        
       # 5. Extract and display answer 
        answer = response.choices[0].message.content
        display(Markdown(answer))

        # 6. Append answer to messages
        messages.append({"role": "assistant", "content": answer})

### Start Tutor Function
The `start_tutor()` function handles all setup before the conversation begins:
- Displays welcome message and available options
- Takes model choice input — sets correct client and model based on selection
- Takes role choice input — selects corresponding system prompt from dictionary
- Validates inputs at every step — exits gracefully if user types 'quit'
- Passes chosen model, client and role to the `chat()` function to begin the session

In [ ]:
def start_tutor():
    # 1. Print welcome message
    print("Welcome to the Technical Interview Tutor!")
    print("Ask me any technical question and I'll help you prepare for your interview!")
    print("Type 'quit' to end the conversation.")
    
    # 2. Show model choices first
    print("\nChoose your model:")
    print("1. GPT-4.1-mini (OpenAI - fast)")
    print("2. Qwen2.5-coder (Ollama - free, slower)")
    model_choice = input("Enter 1 or 2: ")

    if model_choice.lower() == "quit":
        print("Goodbye!")
        return

    # 3. Set client and model based on choice
    if model_choice == "1":
        client = openai_client
        model = MODEL_GPT
        print(f"\nUsing GPT-4.1-mini!")
    else:
        client = ollama_client
        model = MODEL_OLLAMA
        print(f"\nUsing Qwen2.5-coder!")

    # 4. Role choices
    print("\nWhich role are you preparing for?")
    print("1. Data Scientist")
    print("2. Machine Learning Engineer")
    print("3. Business Analyst")
    print("4. Data Analyst")
    print("5. AI/LLM Engineer")
    role_choice = input("Enter 1-5: ")
    if role_choice.lower() == "quit":
        print("Goodbye!")
        return
   
    role_names = {
    "1": "Data Scientist",
    "2": "Machine Learning Engineer",
    "3": "Business Analyst",
    "4": "Data Analyst",
    "5": "AI/LLM Engineer"
    }
    print(f"\nPreparing you for: {role_names[role_choice]} role!")
    print(f"Let's get started!\n")  
        
    # 5. Call chat() with chosen model, client and role
    chat(model, client, role_choice)

# Start the tutor
start_tutor()
